In [1]:
import mlflow
from mlflow.tracking import MlflowClient

# We don't have to set MLflow tracking URI because we set it in an environment variable
# mlflow.set_tracking_uri("http://A.B.C.D:8000/") 

client = MlflowClient()

In [2]:
experiment = client.get_experiment_by_name("loan-risk-xgboost")
experiment

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1746921146045, experiment_id='1', last_update_time=1746921146045, lifecycle_stage='active', name='loan-risk-xgboost', tags={}>

In [3]:
runs = client.search_runs(experiment_ids=[experiment.experiment_id], 
    order_by=["metrics.test_accuracy DESC"], 
    max_results=2)

In [4]:
runs

[<Run: data=<RunData: metrics={'avg_kfold_accuracy': 0.8661075765982632,
  'fold_1_accuracy': 0.866054569029629,
  'fold_2_accuracy': 0.8660628630355509,
  'fold_3_accuracy': 0.866184508455739,
  'fold_4_accuracy': 0.866192802461661,
  'fold_5_accuracy': 0.8660431400087364}, params={'learning_rate': '0.1',
  'max_depth': '6',
  'n_estimators': '100',
  'n_splits': '5'}, tags={'mlflow.runName': 'xgb-kfold-run',
  'mlflow.source.name': '/opt/conda/lib/python3.12/site-packages/ipykernel_launcher.py',
  'mlflow.source.type': 'LOCAL',
  'mlflow.user': 'jovyan'}>, info=<RunInfo: artifact_uri='mlflow-artifacts:/1/3e71ddddda1e4b0188eb695306acc0d9/artifacts', end_time=1746927470799, experiment_id='1', lifecycle_stage='active', run_id='3e71ddddda1e4b0188eb695306acc0d9', run_name='xgb-kfold-run', run_uuid='3e71ddddda1e4b0188eb695306acc0d9', start_time=1746927240404, status='FINISHED', user_id='jovyan'>, inputs=<RunInputs: dataset_inputs=[]>>,
 <Run: data=<RunData: metrics={'val_accuracy': 0.86608

In [8]:
best_run = runs[0]  # The first run is the best due to sorting
best_run_id = best_run.info.run_id
best_test_accuracy = best_run.data.metrics["avg_kfold_accuracy"]
model_uri = f"runs:/{best_run_id}/model"

print(f"Best Run ID: {best_run_id}")
print(f"Test Accuracy: {best_test_accuracy}")
print(f"Model URI: {model_uri}")

Best Run ID: 3e71ddddda1e4b0188eb695306acc0d9
Test Accuracy: 0.8661075765982632
Model URI: runs:/3e71ddddda1e4b0188eb695306acc0d9/model


In [9]:
model_name = "loan-fold5-run1"
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"Model registered as '{model_name}', version {registered_model.version}")

Successfully registered model 'loan-fold5-run1'.
2025/05/11 01:47:17 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: loan-fold5-run1, version 1


Model registered as 'loan-fold5-run1', version 1


Created version '1' of model 'loan-fold5-run1'.
